In [35]:
!pip install catboost

In [26]:
import pandas as pd
import numpy as np


from xgboost import XGBClassifier 
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier , AdaBoostClassifier

In [27]:
df = pd.read_csv("/Users/aayushkamble/ml_pipeline/src/notebook/data/worldcup_cleaned.csv")
df.head()

,Unnamed: 0,Year,Stage,Stadium,City,Home Team Name,Home Team Goals,Away Team Goals,Away Team Name,Attendance,...,Date,month,year,time,Home_Team_Avg_Goals_Scored,Away_Team_Avg_Goals_Scored,Home_Team_Win_Rate,Away_Team_Win_Rate,Head_to_Head_Wins,Is_Home_Advantage
0,0,1930.0,Group 1,Pocitos,Montevideo,France,NaN,1.0,Mexico,4444.0,...,13.0,7.0,1930.0,15:00:00,1.811033,1.0223,0.57277,0.204225,Home Win,1
1,1,1930.0,Group 4,Parque Central,Montevideo,USA,3.0,0.0,Belgium,18346.0,...,13.0,7.0,1930.0,15:00:00,1.811033,1.0223,0.57277,0.204225,Home Win,1
2,2,1930.0,Group 2,Parque Central,Montevideo,Yugoslavia,2.0,1.0,Brazil,24059.0,...,14.0,7.0,1930.0,12:45:00,1.811033,1.0223,0.57277,0.204225,Home Win,1
3,3,1930.0,Group 3,Pocitos,Montevideo,Romania,3.0,1.0,Peru,2549.0,...,14.0,7.0,1930.0,14:50:00,1.811033,1.0223,0.57277,0.204225,Home Win,1
4,4,1930.0,Group 1,Parque Central,Montevideo,Argentina,1.0,0.0,France,23409.0,...,15.0,7.0,1930.0,16:00:00,1.811033,1.0223,0.57277,0.204225,Home Win,1


In [36]:
import pandas as pd

df = pd.read_csv('worldcup_cleaned.csv')
df = df.drop(columns=['Unnamed: 0'])
df = df[df['Year'].notnull()].reset_index(drop=True)

import numpy as np
conditions = [
    df['Home Team Goals'] > df['Away Team Goals'],
    df['Away Team Goals'] > df['Home Team Goals'],
    df['Home Team Goals'] == df['Away Team Goals']
]
df['Result'] = np.select(conditions, ['Home Win','Away Win','Draw'] , default="Draw")

leak_cols = ['Home Team Goals', 'Away Team Goals', 'Is_Home_Advantage',
             'Head_to_Head_Wins', 'Result', 'year',
             'Home Team Initials', 'Away Team Initials', 'time',
             'Stadium', 'City']

X = df.drop(columns=[c for c in leak_cols if c in df.columns])
y = df['Result']

print(type(X))      
print(X.dtypes)

<class 'pandas.core.frame.DataFrame'>
Year                          float64
Stage                          object
Home Team Name                 object
Away Team Name                 object
Attendance                    float64
Date                          float64
month                         float64
Home_Team_Avg_Goals_Scored    float64
Away_Team_Avg_Goals_Scored    float64
Home_Team_Win_Rate            float64
Away_Team_Win_Rate            float64
dtype: object


In [29]:
X = pd.DataFrame(X)

In [30]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# 1. Preprocessing Pipeline for X (Handles NaNs properly)
numerical = X.select_dtypes(exclude='object').columns
categorical = X.select_dtypes(include='object').columns

numerical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessing = ColumnTransformer(
    transformers=[
        ("num", numerical_pipeline, numerical),
        ("cat", categorical_pipeline, categorical)
    ]
)

# Apply Preprocessing to X
X_processed = preprocessing.fit_transform(X)

# 2. Label Encode y for XGBoost (Converts 'Home Win', etc. to 0, 1, 2)
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# 3. Train/Test Split (Crucial: use X_processed and y_encoded)
X_train, X_test, y_train, y_test = train_test_split(X_processed, y_encoded, test_size=0.2, random_state=42)

# 4. Helper function for Classification metrics
def evaluate_model(true, predicted):
    acc = accuracy_score(true, predicted)
    prec = precision_score(true, predicted, average='weighted', zero_division=0)
    rec = recall_score(true, predicted, average='weighted', zero_division=0)
    f1 = f1_score(true, predicted, average='weighted', zero_division=0)
    return acc, prec, rec, f1

# 5. Define Models using a Dictionary
models = {
    "Logistic_regression": LogisticRegression(max_iter=1000), # max_iter increased to prevent warnings
    "RandomForestClassifier": RandomForestClassifier(),
    "XGBClassifier": XGBClassifier()
}

model_list = []
accuracy_list = []

# 6. Train and Evaluate
for model_name, model in models.items():
    print(f"--- Training {model_name} ---")
    
    # Train
    model.fit(X_train, y_train)

    # Predict
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    
    # Evaluate
    train_acc, train_prec, train_rec, train_f1 = evaluate_model(y_train, y_train_pred)
    test_acc, test_prec, test_rec, test_f1 = evaluate_model(y_test, y_test_pred)

    model_list.append(model_name)
    accuracy_list.append(test_acc)

    print(f"Test Dataset Performance:")
    print(f"Accuracy : {test_acc:.4f}")
    print(f"Precision: {test_prec:.4f}")
    print(f"Recall   : {test_rec:.4f}")
    print(f"F1 Score : {test_f1:.4f}\n")


--- Training Logistic_regression ---
Test Dataset Performance:
Accuracy : 0.4795
Precision: 0.4765
Recall   : 0.4795
F1 Score : 0.4732

--- Training RandomForestClassifier ---
Test Dataset Performance:
Accuracy : 0.4737
Precision: 0.4722
Recall   : 0.4737
F1 Score : 0.4574

--- Training XGBClassifier ---


OMP: Warning #179: Function Can't set size of /tmp file failed:


Test Dataset Performance:
Accuracy : 0.4386
Precision: 0.4347
Recall   : 0.4386
F1 Score : 0.4348



In [31]:
X = preprocessing.fit_transform(X)

In [32]:
X.shape

(852, 192)

In [37]:
df.to_csv("worldcup_claeaned2.csv")